# 01 图像检索任务

本 notebook 完成：

1. 读取 `image_retrieval/base/BJTU` 和 `image_retrieval/base/util_pic` 的全部图片作为检索库。
2. 读取 `image_retrieval/query` 的图片作为查询图片。
3. 不使用标签训练，只根据图片内容提取特征并计算相似度。
4. 检索完成后，才根据文件名前缀计算 P@20、P@40、P@60。
5. 输出每类 landmark 的 P@K 图，共 12 张。

注意：`fhy/jx/...` 这些地点标签只在评价阶段使用，不能参与检索排序。

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

from retrieval_utils import (
    LANDMARKS, list_images, build_feature_matrix, retrieve_topk,
    compute_precision_tables, save_all_topk_grids, save_pk_plots,retrieve_topk_rerank
)

IMAGE_RETRIEVAL_ROOT = PROJECT_ROOT / "image_retrieval"
BASE_DIR = IMAGE_RETRIEVAL_ROOT / "base"
QUERY_DIR = IMAGE_RETRIEVAL_ROOT / "query"

OUT_DIR = PROJECT_ROOT / "outputs"
CACHE_DIR = OUT_DIR / "cache"
RETRIEVAL_OUT = OUT_DIR / "retrieval_results"
PK_OUT = OUT_DIR / "pk_curves"
TOPK_GRID_OUT = RETRIEVAL_OUT / "topk_grids"

TOPK = 60
KS = (20, 40, 60)

for p in [CACHE_DIR, RETRIEVAL_OUT, PK_OUT, TOPK_GRID_OUT]:
    p.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Base dir:", BASE_DIR)
print("Query dir:", QUERY_DIR)
print("Landmarks:", LANDMARKS)

Project root: d:\CODE\PY\BJTU_Vision
Base dir: d:\CODE\PY\BJTU_Vision\image_retrieval\base
Query dir: d:\CODE\PY\BJTU_Vision\image_retrieval\query
Landmarks: ['fhy', 'jx', 'kx', 'mh', 'nm', 'sjz', 'sy', 'tsg', 'ty', 'yf', 'yk', 'zx']


In [2]:
base_images = list_images(BASE_DIR)
query_images = list_images(QUERY_DIR)

print("Base images:", len(base_images))
print("Query images:", len(query_images))
print("First 5 base images:")
for p in base_images[:5]:
    print(" ", p)
print("First 5 query images:")
for p in query_images[:5]:
    print(" ", p)

assert len(base_images) > 0, "No base images found. Please check image_retrieval/base."
assert len(query_images) > 0, "No query images found. Please check image_retrieval/query."

Base images: 7728
Query images: 135
First 5 base images:
  d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJTU\fhy-0120240508151840.jpg
  d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJTU\fhy-02wfiftgcgghjxr.jpg
  d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJTU\fhy-04qgjnsjosp902psvnz.png
  d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJTU\fhy-0589095011.png
  d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJTU\fhy-08ur03qu0ridjewifi.jpg
First 5 query images:
  d:\CODE\PY\BJTU_Vision\image_retrieval\query\fhy-12k4jb1k421.jpg
  d:\CODE\PY\BJTU_Vision\image_retrieval\query\fhy-20250506194938.jpg
  d:\CODE\PY\BJTU_Vision\image_retrieval\query\fhy-202505071933041.jpg
  d:\CODE\PY\BJTU_Vision\image_retrieval\query\fhy-20260511140018_130_105(1).jpg
  d:\CODE\PY\BJTU_Vision\image_retrieval\query\fhy-2837bfg872vd.jpg


## 提取特征

这里使用 OpenCV 手工图像特征：HSV 颜色直方图 + 灰度缩略图 + 边缘缩略图。它不需要 PyTorch，不需要训练，也不会使用标签，速度比较快。

In [3]:
base_paths, base_features = build_feature_matrix(
    base_images,
    cache_path=CACHE_DIR / "base_features.npz",
    force_rebuild=True
)
query_paths, query_features = build_feature_matrix(
    query_images,
    cache_path=CACHE_DIR / "query_features.npz",
    force_rebuild=True
)

print("Base feature matrix:", base_features.shape)
print("Query feature matrix:", query_features.shape)

Extracting image features: 100%|██████████| 7728/7728 [01:32<00:00, 83.41it/s] 


Skipped 1 unreadable images. See: d:\CODE\PY\BJTU_Vision\outputs\cache\base_features_skipped_images.csv


Extracting image features: 100%|██████████| 135/135 [00:02<00:00, 58.91it/s]

Base feature matrix: (7727, 2560)
Query feature matrix: (135, 2560)


## 进行 TopK 检索

这一步只用特征相似度排序。`query_label` 和 `base_label` 是检索完成后为了计算准确率才提取的。

In [4]:
topk_df = retrieve_topk_rerank(
    query_paths,
    query_features,
    base_paths,
    base_features,
    first_stage_k=500,
    final_k=TOPK,
    alpha=0.35
)
retrieval_csv = RETRIEVAL_OUT / "retrieval_topk.csv"
topk_df.to_csv(retrieval_csv, index=False, encoding="utf-8-sig")
print("Saved:", retrieval_csv)
topk_df.head(10)

Preparing ORB descriptors...


Retrieving with ORB rerank: 100%|██████████| 135/135 [02:51<00:00,  1.27s/it]

Saved: d:\CODE\PY\BJTU_Vision\outputs\retrieval_results\retrieval_topk.csv


,query_path,query_name,query_label,base_path,base_name,base_label,global_similarity,orb_similarity,similarity,relevant,rank
0,d:\CODE\PY\BJTU_Vision\image_retrieval\query\f...,fhy-12k4jb1k421.jpg,fhy,d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJ...,tsg-nnvovovoovoo.jpg,tsg,0.487390,0.1250,0.251836,False,1
1,d:\CODE\PY\BJTU_Vision\image_retrieval\query\f...,fhy-12k4jb1k421.jpg,fhy,d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJ...,yk-T6V9X3Z7A2D5G8K1.png,yk,0.388156,0.1625,0.241479,False,2
2,d:\CODE\PY\BJTU_Vision\image_retrieval\query\f...,fhy-12k4jb1k421.jpg,fhy,d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJ...,zx-oitytf.jpg,zx,0.410126,0.1500,0.241044,False,3
3,d:\CODE\PY\BJTU_Vision\image_retrieval\query\f...,fhy-12k4jb1k421.jpg,fhy,d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJ...,ty-1evb87.jpg,ty,0.520060,0.0875,0.238896,False,4
4,d:\CODE\PY\BJTU_Vision\image_retrieval\query\f...,fhy-12k4jb1k421.jpg,fhy,d:\CODE\PY\BJTU_Vision\image_retrieval\base\ut...,worcester_000114.jpg,irrelevant,0.496336,0.1000,0.238718,False,5
5,d:\CODE\PY\BJTU_Vision\image_retrieval\query\f...,fhy-12k4jb1k421.jpg,fhy,d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJ...,nm-p93rqhnksp3249.png,nm,0.448024,0.1250,0.238058,False,6
6,d:\CODE\PY\BJTU_Vision\image_retrieval\query\f...,fhy-12k4jb1k421.jpg,fhy,d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJ...,sjz-adf100.jpg,sjz,0.374754,0.1625,0.236789,False,7
7,d:\CODE\PY\BJTU_Vision\image_retrieval\query\f...,fhy-12k4jb1k421.jpg,fhy,d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJ...,zx-sbdfbdgfbh.jpg,zx,0.388269,0.1500,0.233394,False,8
8,d:\CODE\PY\BJTU_Vision\image_retrieval\query\f...,fhy-12k4jb1k421.jpg,fhy,d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJ...,sjz-jhbhbhsy.jpg,sjz,0.377631,0.1500,0.229671,False,9
9,d:\CODE\PY\BJTU_Vision\image_retrieval\query\f...,fhy-12k4jb1k421.jpg,fhy,d:\CODE\PY\BJTU_Vision\image_retrieval\base\BJ...,sjz-20160428144501870536.jpg,sjz,0.377631,0.1500,0.229671,False,10


## 计算 Precision@K

P@K = 前 K 个检索结果中，与 query 属于同一个地点的图片比例。

In [5]:
per_query, per_label = compute_precision_tables(topk_df, ks=KS)
per_query_csv = RETRIEVAL_OUT / "precision_by_query.csv"
per_label_csv = RETRIEVAL_OUT / "precision_by_label.csv"
per_query.to_csv(per_query_csv, index=False, encoding="utf-8-sig")
per_label.to_csv(per_label_csv, index=False, encoding="utf-8-sig")
print("Saved:", per_query_csv)
print("Saved:", per_label_csv)
per_label

Saved: d:\CODE\PY\BJTU_Vision\outputs\retrieval_results\precision_by_query.csv
Saved: d:\CODE\PY\BJTU_Vision\outputs\retrieval_results\precision_by_label.csv


,label,num_queries,P@20,P@40,P@60
0,fhy,15,0.626667,0.545000,0.428889
1,jx,7,0.478571,0.389286,0.304762
2,kx,2,0.050000,0.025000,0.025000
3,mh,14,0.678571,0.487500,0.366667
4,nm,15,0.173333,0.128333,0.113333
5,sjz,15,0.310000,0.253333,0.232222
6,sy,15,0.596667,0.445000,0.344444
7,tsg,15,0.313333,0.215000,0.175556
8,ty,14,0.560714,0.464286,0.375000
9,yf,7,0.100000,0.067857,0.069048


## 生成每类 landmark 的 P@K 图

此处生成 12 张图，对应 12 个地点。

In [6]:
pk_paths = save_pk_plots(per_query, per_label, PK_OUT, ks=KS)
print("Generated P@K plots:", len(pk_paths))
for p in pk_paths:
    print(p)

Generated P@K plots: 12
d:\CODE\PY\BJTU_Vision\outputs\pk_curves\fhy_pk.png
d:\CODE\PY\BJTU_Vision\outputs\pk_curves\jx_pk.png
d:\CODE\PY\BJTU_Vision\outputs\pk_curves\kx_pk.png
d:\CODE\PY\BJTU_Vision\outputs\pk_curves\mh_pk.png
d:\CODE\PY\BJTU_Vision\outputs\pk_curves\nm_pk.png
d:\CODE\PY\BJTU_Vision\outputs\pk_curves\sjz_pk.png
d:\CODE\PY\BJTU_Vision\outputs\pk_curves\sy_pk.png
d:\CODE\PY\BJTU_Vision\outputs\pk_curves\tsg_pk.png
d:\CODE\PY\BJTU_Vision\outputs\pk_curves\ty_pk.png
d:\CODE\PY\BJTU_Vision\outputs\pk_curves\yf_pk.png
d:\CODE\PY\BJTU_Vision\outputs\pk_curves\yk_pk.png
d:\CODE\PY\BJTU_Vision\outputs\pk_curves\zx_pk.png


## 生成 TopK 检索展示图

`topk_grids` 里面保存的是检索过程展示图：每张图左边是 Query，后面是 Top1、Top2、... 检索结果。绿色表示该检索结果与 query 同类，红色表示不同类。

In [7]:
selected_queries = save_all_topk_grids(topk_df, TOPK_GRID_OUT, k=10, max_per_label=2)
print("Saved TopK demo grids:", len(selected_queries))
print("Output dir:", TOPK_GRID_OUT)

Saving TopK grids: 100%|██████████| 24/24 [00:07<00:00,  3.40it/s]

Saved TopK demo grids: 24
Output dir: d:\CODE\PY\BJTU_Vision\outputs\retrieval_results\topk_grids
